In [2]:
import spacy
nlp = spacy.load("en_core_web_md")
print("spaCy 模型加载成功！")

from sentence_transformers import SentenceTransformer, util
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')
print("SentenceTransformer 模型加载成功！")


TypeError: ForwardRef._evaluate() missing 1 required keyword-only argument: 'recursive_guard'

In [6]:
import json
import re
import numpy as np
import pandas as pd
import random
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

##############################################################################
# 0.1 Try to import spaCy for semantic matching
##############################################################################
try:
    import spacy
    nlp = spacy.load("en_core_web_md")
    USE_SPACY = True
    print("✅ Using spaCy for semantic matching.")
except ModuleNotFoundError:
    print("⚠️ spaCy not found, fallback to simple methods.")
    USE_SPACY = False

try:
    from sentence_transformers import SentenceTransformer, util
    sbert_model = SentenceTransformer('all-MiniLM-L6-v2')
    USE_SBERT = True
    print("✅ Using SentenceTransformer for semantic matching.")
except ModuleNotFoundError:
    print("⚠️ SentenceTransformer not found, fallback to simple methods.")
    USE_SBERT = False

##############################################################################
# 1. Load JSON Data & Preprocessing
##############################################################################
file_path = "company_product_data.json"
file_path1 = "patents.json"

with open(file_path, "r", encoding="utf-8") as f:
    company_product_data = json.load(f)
with open(file_path1, "r", encoding="utf-8") as f:
    patents = json.load(f)  # Read company product data and patent data.

def preprocess_text(text):
    """
    Text preprocessing: lowercase, remove special characters, numbers & stopwords.
    """
    text = text.lower()  # Converts all text to lowercase
    text = re.sub(r'[^a-z\s]', '', text)  # Keep only letters
    stopwords = {'the','and','with','to','for','of','a','in','on','at','from','by','this','that','these','those','be','an','or'}
    tokens = [w for w in text.split() if w not in stopwords]  # Remove common stop words
    return ' '.join(tokens)

# Process products
cleaned_products = []
for comp in company_product_data["companies"]:
    for prod in comp["products"]:
        desc = prod.get("description", "")
        cleaned_desc = preprocess_text(desc)
        cleaned_products.append({
            "company": comp["name"],
            "product_number": prod.get("product_id", "Unknown"),  # Add product number
            "product_name": prod["name"],
            "cleaned_description": cleaned_desc
        })
cleaned_product_df = pd.DataFrame(cleaned_products)
product_names = cleaned_product_df["product_name"].tolist()
product_numbers = cleaned_product_df["product_number"].tolist()
product_texts = cleaned_product_df["cleaned_description"].tolist()
num_products = len(product_texts)

# Process patents (combine abstract + claims into merged_text)
cleaned_patents = []
for pat in patents:
    abstract = pat.get("abstract", "")
    claims   = pat.get("claims", [])
    if isinstance(claims, list):
        claims_text = " ".join([c.get("text", "") for c in claims])
    else:
        claims_text = ""
    merged_text = preprocess_text(abstract + " " + claims_text) 
    # Extract the abstract and claims of the patent
    
    cleaned_patents.append({
        "publication_number": pat["publication_number"],
        "cleaned_claims": preprocess_text(claims_text),
        "merged_text": merged_text
    })
cleaned_patent_df = pd.DataFrame(cleaned_patents)
patent_numbers = cleaned_patent_df["publication_number"].tolist()
patent_merged  = cleaned_patent_df["merged_text"].tolist()
num_patents = len(patent_merged)

print(f"📌 Products: {num_products}, Patents: {num_patents}")

# Build a dictionary for patent claims (for subsequent extraction of "Relevant Claims")
patent_claims_dict = dict(zip(patent_numbers, cleaned_patent_df["cleaned_claims"]))

##############################################################################
# 2. TF-IDF Vectorization + Cosine Similarity
##############################################################################
max_features = 5000  # Restricted feature dimension
tfidf = TfidfVectorizer(stop_words='english', max_features=max_features)
corpus = product_texts + patent_merged
tfidf_matrix = tfidf.fit_transform(corpus) 
# tfidf_matrix is divided into two parts: pre-num_products product vector and post-num_patents patent vector.

product_vectors = tfidf_matrix[:num_products]
patent_vectors  = tfidf_matrix[num_products:num_products+num_patents]

# Calculate cosine similarity using TF-IDF vectors (sparse matrix)
cosine_sim = cosine_similarity(product_vectors, patent_vectors)
# Build a DataFrame with product names as row index and patent numbers as column index
product_patent_sim_df = pd.DataFrame(cosine_sim, index=product_names, columns=patent_numbers)

##############################################################################
# 3. Retrieve Matches & Compute Alignment Keywords
##############################################################################
top_k = 15 # Number of candidate key words                              
pairs_list = []

for p_idx in range(num_products):
    # Take the similarity vector of the product to all patents
    best_match_idx = np.argmax(cosine_sim[p_idx])
    best_sim_score = cosine_sim[p_idx][best_match_idx]
    
    pairs_list.append((p_idx, best_match_idx, best_sim_score))

all_pairs_df = pd.DataFrame(pairs_list, columns=["ProductIdx", "PatentIdx", "SimilarityScore"])

# Add product name, product number, and patent number columns
all_pairs_df["Product Name"] = all_pairs_df["ProductIdx"].apply(lambda i: product_names[i])
all_pairs_df["Product Number"] = all_pairs_df["ProductIdx"].apply(lambda i: product_numbers[i])
all_pairs_df["Patent Number"] = all_pairs_df["PatentIdx"].apply(lambda i: patent_numbers[i])

##############################################################################
# 4. Compute Infringement Labels & ML-based Infringement Prediction
##############################################################################
# Simple threshold: if similarity > 0.5, mark as potential infringement (label=1) 
all_pairs_df["InfringementLabel"] = (all_pairs_df["SimilarityScore"] > 0.5).astype(int) 
# Similarity threshold = 0.5: Looser matching can be guaranteed to facilitate subsequent manual audits

# For speed, use threshold result as ML-based infringement prediction
all_pairs_df["ML-based Infringement Prediction"] = all_pairs_df["InfringementLabel"]

##############################################################################
# 5. Extract Relevant Claims using Semantic Matching with spaCy
##############################################################################
def extract_relevant_claims_semantic(p_idx, patent_num, top_k=5, sim_threshold=0.5):
    """
spaCy is used to calculate the semantic similarity of each word in the product description and patent claims.
Returns top_k words in the patent claim that are semantically similar to the product description (similarity >=sim_threshold).
If spaCy is not available, fall back on the simple public keyword method.

    """
    if patent_num not in patent_claims_dict:
        return "No matching relevant words"
    
    product_txt = product_texts[p_idx]
    claims_txt = patent_claims_dict[patent_num]
    if not product_txt.strip() or not claims_txt.strip():
        return "No matching relevant words"
    
    
    if USE_SBERT:
            # 按句切分文本
            prod_sentences = re.split(r'(?<=[.!?])\s+', product_txt)
            claim_sentences = re.split(r'(?<=[.!?])\s+', claims_txt)
            prod_sentences = [s.strip() for s in prod_sentences if s.strip()]
            claim_sentences = [s.strip() for s in claim_sentences if s.strip()]
            if not prod_sentences or not claim_sentences:
                return "No matching relevant words"
            prod_embeddings = sbert_model.encode(prod_sentences, convert_to_tensor=True)
            claim_embeddings = sbert_model.encode(claim_sentences, convert_to_tensor=True)
            cosine_scores = cosine_similarity(prod_embeddings, claim_embeddings)
            # 对于每个 claim 句子，取其与产品描述中最高的相似度
            max_scores = np.max(cosine_scores, axis=0)
            selected_indices = np.where(max_scores >= sim_threshold)[0]
            if len(selected_indices) == 0:
                return "No matching relevant words"
            selected = sorted(selected_indices, key=lambda i: max_scores[i], reverse=True)[:top_k]
            top_sentences = [claim_sentences[i] for i in selected]
            return " || ".join(top_sentences)
    elif USE_SPACY:
        prod_doc = nlp(product_txt)
        claims_doc = nlp(claims_txt)
        prod_tokens = [token for token in prod_doc if not token.is_stop and not token.is_punct and token.has_vector]
        claim_tokens = [token for token in claims_doc if not token.is_stop and not token.is_punct and token.has_vector]
        token_sim_list = []
        for token in claim_tokens:
            max_sim = max((token.similarity(pt) for pt in prod_tokens), default=0)
            if max_sim >= sim_threshold:
                token_sim_list.append((token.text, max_sim))
        if not token_sim_list:
            return "No matching relevant words"
        token_sim_list.sort(key=lambda x: x[1], reverse=True)
        top_tokens = [token for token, sim in token_sim_list[:top_k]]
        return ", ".join(top_tokens)
    else:
        prod_words = set(product_txt.split())
        claim_words = set(claims_txt.split())
        common_words = prod_words.intersection(claim_words)
        if not common_words:
            return "No matching common words"
        return ", ".join(list(common_words)[:top_k])

def extract_relevant_claims(p_idx, patent_num, top_k=10):
    return extract_relevant_claims_semantic(p_idx, patent_num, top_k=top_k, sim_threshold=0.5)

# 构建映射字典
product_idx_to_name = dict(enumerate(product_names))
patent_idx_to_num = dict(enumerate(patent_numbers))

all_pairs_df["Product Name"] = all_pairs_df["ProductIdx"].apply(lambda i: product_idx_to_name.get(i, "UnknownProd"))
all_pairs_df["Patent Number"] = all_pairs_df["PatentIdx"].apply(lambda i: patent_idx_to_num.get(i, "UnknownPat"))

all_pairs_df["Relevant Claims"] = all_pairs_df.apply(
    lambda row: extract_relevant_claims(int(row["ProductIdx"]), row["Patent Number"], top_k=10),
    axis=1
)



##############################################################################
# 6. Final Output and Suspected Infringement Output
##############################################################################
final_output = all_pairs_df[[
    "Product Name",
    "Patent Number",
    "SimilarityScore",
    "ML-based Infringement Prediction",
    "Relevant Claims"
]]

print("\n📌 **Final Output (Sample)**:")
print(final_output.head(15))

Suspected_infringement_output = all_pairs_df[all_pairs_df["SimilarityScore"] > 0.5][[
    "Product Name",
    "Patent Number", # the patent that is the most similar to the corresponding product
    "SimilarityScore",
    "ML-based Infringement Prediction", # 1: suspected infringement, 0: not infringement
    "Relevant Claims" # Words in the product description that relevant to the claims and abstracts of the patent.
]].copy()

print("\n--- Suspected_infringement_output (SimilarityScore > 0.5) ---")
print(Suspected_infringement_output.head(15))


final_output.to_csv("filtered_product_patent_output.csv", index=False)



TypeError: ForwardRef._evaluate() missing 1 required keyword-only argument: 'recursive_guard'

In [1]:
import spacy


TypeError: ForwardRef._evaluate() missing 1 required keyword-only argument: 'recursive_guard'